У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
import pandas as pd
import numpy as np

# Завантажимо дані
df = pd.read_csv('customer_segmentation_train.csv')
df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [3]:
# Розподіл цільової змінної
print(df['Segmentation'].value_counts())
print()

# Кількість унікальних значень у категоріальних ознаках
for col in ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score', 'Var_1']:
    print(col, ':', df[col].nunique(), '->', df[col].unique())

Segmentation
D    2268
A    1972
C    1970
B    1858
Name: count, dtype: int64

Gender : 2 -> ['Male' 'Female']
Ever_Married : 2 -> ['No' 'Yes' nan]
Graduated : 2 -> ['No' 'Yes' nan]
Profession : 9 -> ['Healthcare' 'Engineer' 'Lawyer' 'Entertainment' 'Artist' 'Executive'
 'Doctor' 'Homemaker' 'Marketing' nan]
Spending_Score : 3 -> ['Low' 'Average' 'High']
Var_1 : 7 -> ['Cat_4' 'Cat_6' 'Cat_7' 'Cat_3' 'Cat_1' 'Cat_2' nan 'Cat_5']


In [4]:
# Приберемо службову колонку ID
df = df.drop(columns='ID')

cat_features = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score', 'Var_1']
num_features = ['Age', 'Work_Experience', 'Family_Size']

# Заповнимо пропуски: категоріальні - модою, числові - медіаною
for col in cat_features:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in num_features:
    df[col] = df[col].fillna(df[col].median())

df.isna().sum()

,0
Gender,0
Ever_Married,0
Age,0
Graduated,0
Profession,0
Work_Experience,0
Spending_Score,0
Family_Size,0
Var_1,0
Segmentation,0


In [5]:
from sklearn.preprocessing import OrdinalEncoder

# Закодуємо категоріальні ознаки (по одній колонці на ознаку - потрібно для SMOTENC)
encoder = OrdinalEncoder()
df[cat_features] = encoder.fit_transform(df[cat_features])

target_col_name = 'Segmentation'
X = df.drop(columns=target_col_name)
y = df[target_col_name]

X.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
0,1.0,0.0,22,0.0,5.0,1.0,2.0,4.0,3.0
1,0.0,1.0,38,1.0,2.0,1.0,0.0,3.0,3.0
2,0.0,1.0,67,1.0,2.0,1.0,2.0,1.0,5.0
3,1.0,1.0,67,1.0,7.0,0.0,1.0,2.0,5.0
4,0.0,1.0,40,1.0,3.0,1.0,1.0,6.0,5.0


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Розділимо дані на тренувальні та тестові набори
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the numeric features.
# Note that we fit MinMaxScaler on X_train only, not on the entire dataset.
scaler = MinMaxScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print(X_train.shape, X_test.shape)
print(y_train.value_counts())

(6454, 9) (1614, 9)
Segmentation
D    1814
A    1578
C    1576
B    1486
Name: count, dtype: int64


### Висновки до Завдання 1

Датасет містить 8068 записів та 10 ознак (службову колонку `ID` видалено).

- **Пропуски** були в 5 колонках, найбільше - у `Work_Experience` (829 значень). Категоріальні ознаки заповнено модою, числові - медіаною.
- **Категоріальні ознаки** (`Gender`, `Ever_Married`, `Graduated`, `Profession`, `Spending_Score`, `Var_1`) закодовано через `OrdinalEncoder`. Обрано саме ordinal-кодування, а не one-hot, бо для методу `SMOTENC` потрібні індекси категоріальних колонок - одна колонка на ознаку.
- **Числові ознаки** (`Age`, `Work_Experience`, `Family_Size`) масштабовано за допомогою `MinMaxScaler`.
- Дані розбито у співвідношенні 80/20 зі `stratify=y`: 6454 записи в тренувальній вибірці та 1614 — у тестовій.


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [7]:
from imblearn.over_sampling import SMOTE

# Варіант 1: базовий SMOTE лише на некатегоріальних ознаках
smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train[num_features], y_train)

print(X_train_smote.shape)
print(y_train_smote.value_counts())

(7256, 3)
Segmentation
A    1814
B    1814
C    1814
D    1814
Name: count, dtype: int64


In [8]:
from imblearn.over_sampling import SMOTENC

# Варіант 2: SMOTENC - працює і з категоріальними ознаками
cat_feature_indeces = [X_train.columns.get_loc(col) for col in cat_features]
print('Індекси категоріальних ознак:', cat_feature_indeces)

smotenc = SMOTENC(categorical_features=cat_feature_indeces, random_state=0)
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train, y_train)

print(X_train_smotenc.shape)
print(y_train_smotenc.value_counts())

Індекси категоріальних ознак: [0, 1, 3, 4, 6, 8]
(7256, 9)
Segmentation
A    1814
B    1814
C    1814
D    1814
Name: count, dtype: int64


In [9]:
from imblearn.combine import SMOTETomek

# Ресемплінг з SMOTE-Tomek (всередині використовуємо SMOTENC для коректної роботи з категоріями)
smotetomek = SMOTETomek(smote=SMOTENC(categorical_features=cat_feature_indeces, random_state=0),
                        random_state=0)
X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train, y_train)

print(X_train_smotetomek.shape)
print(y_train_smotetomek.value_counts())

(5704, 9)
Segmentation
C    1470
D    1451
B    1404
A    1379
Name: count, dtype: int64


### Висновки до Завдання 2


Отримано три варіанти тренувальних наборів:

1. **Базовий SMOTE** (`X_train_smote`) - застосований лише до числових ознак `Age`, `Work_Experience`, `Family_Size`. Класи вирівняно до 1814 кожен (7256 записів), але набір містить лише 3 ознаки з 9.
2. **SMOTENC** (`X_train_smotenc`) - модифікація SMOTE, яка коректно обробляє категоріальні ознаки: передано індекси категоріальних колонок `cat_feature_indeces = [0, 1, 3, 4, 6, 8]`. Результат — 7256 записів з усіма 9 ознаками та ідеально збалансованими класами.
3. **SMOTE-Tomek** (`X_train_smotetomek`) — комбінація оверсемплінгу та андерсемплінгу. Всередину передано `SMOTENC`, щоб категоріальні ознаки оброблялись коректно.

SMOTE-Tomek видалив 1552 записи з 7256 (тобто 776 пар Томека), залишивши 5704. Після видалення звʼязків Томека класи вже не ідеально збалансовані (від 1379 до 1470), оскільки видаляються точки з обох сторін межі.

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

# Модель 1: OvR на оригінальних даних
log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(X_train, y_train)
ovr_predictions = ovr_model.predict(X_test)

print(classification_report(y_test, ovr_predictions))

              precision    recall  f1-score   support

           A       0.39      0.39      0.39       394
           B       0.41      0.08      0.13       372
           C       0.47      0.64      0.54       394
           D       0.59      0.79      0.67       454

    accuracy                           0.49      1614
   macro avg       0.46      0.48      0.44      1614
weighted avg       0.47      0.49      0.45      1614



In [11]:
# Модель 2: OvR на даних, збалансованих базовим SMOTE (лише числові ознаки)
ovr_model_smote = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
ovr_model_smote.fit(X_train_smote, y_train_smote)
ovr_smote_predictions = ovr_model_smote.predict(X_test[num_features])

print(classification_report(y_test, ovr_smote_predictions))

              precision    recall  f1-score   support

           A       0.33      0.28      0.31       394
           B       0.29      0.10      0.15       372
           C       0.37      0.42      0.39       394
           D       0.50      0.75      0.60       454

    accuracy                           0.41      1614
   macro avg       0.37      0.39      0.36      1614
weighted avg       0.38      0.41      0.37      1614



In [12]:
# Модель 3: OvR на даних, збалансованих SMOTENC (усі ознаки)
ovr_model_smotenc = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
ovr_model_smotenc.fit(X_train_smotenc, y_train_smotenc)
ovr_smotenc_predictions = ovr_model_smotenc.predict(X_test)

print(classification_report(y_test, ovr_smotenc_predictions))

              precision    recall  f1-score   support

           A       0.42      0.40      0.41       394
           B       0.37      0.13      0.19       372
           C       0.47      0.63      0.54       394
           D       0.62      0.78      0.69       454

    accuracy                           0.50      1614
   macro avg       0.47      0.49      0.46      1614
weighted avg       0.48      0.50      0.47      1614



In [13]:
# Модель 4: OvR на даних, збалансованих SMOTE-Tomek
ovr_model_smotetomek = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
ovr_model_smotetomek.fit(X_train_smotetomek, y_train_smotetomek)
ovr_smotetomek_predictions = ovr_model_smotetomek.predict(X_test)

print(classification_report(y_test, ovr_smotetomek_predictions))

              precision    recall  f1-score   support

           A       0.40      0.39      0.39       394
           B       0.33      0.11      0.17       372
           C       0.47      0.62      0.53       394
           D       0.61      0.78      0.68       454

    accuracy                           0.49      1614
   macro avg       0.45      0.48      0.44      1614
weighted avg       0.46      0.49      0.46      1614



In [14]:
from sklearn.metrics import f1_score, accuracy_score

# Створимо датафрейм для відображення результатів
results = pd.DataFrame({
    'Model': ['Original', 'SMOTE (num only)', 'SMOTENC', 'SMOTE-Tomek'],
    'Accuracy': [
        accuracy_score(y_test, ovr_predictions),
        accuracy_score(y_test, ovr_smote_predictions),
        accuracy_score(y_test, ovr_smotenc_predictions),
        accuracy_score(y_test, ovr_smotetomek_predictions)
    ],
    'Macro F1': [
        f1_score(y_test, ovr_predictions, average='macro'),
        f1_score(y_test, ovr_smote_predictions, average='macro'),
        f1_score(y_test, ovr_smotenc_predictions, average='macro'),
        f1_score(y_test, ovr_smotetomek_predictions, average='macro')
    ],
    'F1 class B': [
        f1_score(y_test, ovr_predictions, average=None, labels=['B'])[0],
        f1_score(y_test, ovr_smote_predictions, average=None, labels=['B'])[0],
        f1_score(y_test, ovr_smotenc_predictions, average=None, labels=['B'])[0],
        f1_score(y_test, ovr_smotetomek_predictions, average=None, labels=['B'])[0]
    ]
})
print(results.round(3))

              Model  Accuracy  Macro F1  F1 class B
0          Original     0.493     0.435       0.135
1  SMOTE (num only)     0.408     0.363       0.154
2           SMOTENC     0.502     0.457       0.188
3       SMOTE-Tomek     0.491     0.445       0.168


### Висновки до Завдання 3

**3. Яку метрику обрано для порівняння**

Основна метрика - **macro F1-score**. Обґрунтування:

- Задача багатокласова, і всі 4 сегменти клієнтів однаково важливі для бізнесу - macro-усереднення дає кожному класу однакову вагу незалежно від його розміру.
- F1 поєднує precision і recall, тому враховує як хибнопозитивні, так і хибнонегативні помилки.

**4. Яка модель найкраща**

Найкращий результат показала модель, натренована на даних після **SMOTENC**: macro F1 = 0.457, accuracy = 0.502, F1 для класу B = 0.188.

Окремо відзначу провал базового SMOTE. За умовою його застосовано **лише до 3 числових ознак** - модель втратила 6 категоріальних ознак і разом з ними значну частину інформації. Це показує, чому для змішаних даних потрібен саме SMOTENC.

**5. Гіпотеза, чому різниця між моделями несуттєва**

Різниця між Original, SMOTENC та SMOTE-Tomek становить лише 0.01–0.02 за macro F1, тобто знаходиться в межах статистичного шуму.

**Дисбалансу класів фактично немає.** Співвідношення найбільшого класу D (1814) до найменшого B (1486) у тренувальній вибірці - лише 1.2:1. Методи ресемплінгу створені для більших співвідношень. Коли дисбаланс мінімальний, балансуванню не має на що впливати.

